[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/A-Kuo/Data-Engineering-Fork-of-AmFam-Workshop/blob/main/explorations/fairness_basics_demo.ipynb)

# Fairness basics (synthetic, educational)

**NAIC AI bulletin** emphasizes testing for **unfair discrimination**. This notebook uses **entirely synthetic** data with a fake `group` column — **not** real protected attributes.

**What you learn:** Demographic parity difference, why equalizing approval rates across groups is non-trivial, and how threshold tuning interacts with fairness.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression

np.random.seed(7)
n = 2000
# Synthetic: group A vs B; base fraud rate differs (simulates historical disparity)
group = np.random.binomial(1, 0.5, n)
fraud_true = np.where(
    group == 0,
    np.random.binomial(1, 0.04, n),
    np.random.binomial(1, 0.08, n),
)
claim_amount = np.random.lognormal(8, 1.0, n) * (1 + fraud_true * 0.4)
priors = np.random.poisson(0.7 + fraud_true * 1.5, n).clip(0, 12)
X = np.column_stack([claim_amount, priors, group])
y = fraud_true

clf = LogisticRegression(class_weight='balanced', max_iter=300)
clf.fit(X, y)
proba = clf.predict_proba(X)[:, 1]
df = pd.DataFrame({'group': group, 'fraud': y, 'proba': proba})
df.head()

## Demographic parity at threshold 0.5

P(flagged | group=0) vs P(flagged | group=1) at a fixed threshold.

In [ ]:
thr = 0.5
flagged = (proba >= thr).astype(int)
df['flagged'] = flagged
rates = df.groupby('group')['flagged'].mean()
print('Flag rate group 0:', rates[0].round(4))
print('Flag rate group 1:', rates[1].round(4))
dp_diff = abs(rates[0] - rates[1])
print('Demographic parity difference:', round(dp_diff, 4))

# Sweep thresholds
ths = np.linspace(0.1, 0.9, 50)
g0 = [(proba[group == 0] >= t).mean() for t in ths]
g1 = [(proba[group == 1] >= t).mean() for t in ths]
plt.figure(figsize=(8, 4))
plt.plot(ths, g0, label='group 0 flag rate')
plt.plot(ths, g1, label='group 1 flag rate')
plt.xlabel('threshold')
plt.ylabel('P(flagged)')
plt.legend()
plt.title('Synthetic: flag rates vs threshold by group')
plt.show()

## What we learned

- **Parity is not automatic** — Even with `class_weight`, groups with different base rates can get different flag rates.
- **Threshold per group** (rarely legal in credit/insurance without care) vs **single threshold** vs **constrained optimization** — real compliance needs legal + actuarial review.
- **Interview line:** "I know fairness metrics exist; production changes require governance — this notebook shows I can *measure* disparity before shipping."
- **Link to repo:** Your **tabular fraud** + **eval logging** notebooks are the stack regulators ask to document: **test, log, review**.